## this is a notebook I have found in kaggle for image classification model on this dataset

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# List all files under the local data directory
import os
for dirname, _, filenames in os.walk('./data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!nvidia-smi

In [ ]:
import os
os.listdir("./data")


In [ ]:
import os

base = "./data"
os.listdir(base)


In [ ]:
for item in os.listdir(base):
    print(item)


In [ ]:
# explore RDD_SPLIT folder (local dataset has no country subfolders)
os.listdir(os.path.join(base, "RDD_SPLIT"))


In [ ]:
import os

base = "./data/RDD_SPLIT"
os.listdir(base)


In [ ]:
import os

train_path = "./data/RDD_SPLIT/train"
os.listdir(train_path)


In [ ]:
import os

label_dir = "./data/RDD_SPLIT/train/labels"
label_files = os.listdir(label_dir)

label_files[:5]


In [ ]:
sample_label = os.path.join(label_dir, label_files[0])

with open(sample_label, "r") as f:
    print(f.read())

In [ ]:
# find first non-empty label file
for lf in label_files:
    path = os.path.join(label_dir, lf)
    if os.path.getsize(path) > 0:
        print("Non-empty label file:", lf)
        with open(path, "r") as f:
            print(f.read())
        break


In [ ]:
import os

base_out = "./output/road_yolo"

dirs = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for d in dirs:
    os.makedirs(os.path.join(base_out, d), exist_ok=True)

print("Folders created successfully")


In [ ]:
import os
import shutil
import random

# Source paths (local RDD_SPLIT dataset)
SRC_BASE = "./data/RDD_SPLIT"
SRC_TRAIN_IMG = os.path.join(SRC_BASE, "train/images")
SRC_TRAIN_LBL = os.path.join(SRC_BASE, "train/labels")
SRC_VAL_IMG   = os.path.join(SRC_BASE, "val/images")
SRC_VAL_LBL   = os.path.join(SRC_BASE, "val/labels")

# Destination paths (our small dataset)
DST_BASE = "./output/road_yolo"
DST_TRAIN_IMG = os.path.join(DST_BASE, "images/train")
DST_TRAIN_LBL = os.path.join(DST_BASE, "labels/train")
DST_VAL_IMG   = os.path.join(DST_BASE, "images/val")
DST_VAL_LBL   = os.path.join(DST_BASE, "labels/val")

# Class remapping
CLASS_MAP = {
    0: 0,  # crack
    1: 0,  # crack
    2: 0,  # crack
    3: 1,  # pothole
    4: 2   # patch
}

print("Paths and class mapping ready")


In [ ]:
# number of training samples we want
TRAIN_LIMIT = 8000

# get all training images
all_train_images = sorted(os.listdir(SRC_TRAIN_IMG))
random.shuffle(all_train_images)

selected_train_images = all_train_images[:TRAIN_LIMIT]

copied = 0
skipped = 0

for img_name in selected_train_images:
    img_path = os.path.join(SRC_TRAIN_IMG, img_name)
    lbl_name = img_name.replace(".jpg", ".txt")
    lbl_path = os.path.join(SRC_TRAIN_LBL, lbl_name)

    new_labels = []

    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                orig_cls = int(parts[0])
                if orig_cls in CLASS_MAP:
                    new_cls = CLASS_MAP[orig_cls]
                    new_labels.append(
                        f"{new_cls} {' '.join(parts[1:])}"
                    )

    # copy image
    shutil.copy(img_path, os.path.join(DST_TRAIN_IMG, img_name))

    # write new label file (can be empty)
    with open(os.path.join(DST_TRAIN_LBL, lbl_name), "w") as f:
        for l in new_labels:
            f.write(l + "\n")

    copied += 1

print(f"Training images copied: {copied}")
print(f"Training images skipped: {skipped}")


In [ ]:
# number of validation samples we want
VAL_LIMIT = 2000

# get all validation images
all_val_images = sorted(os.listdir(SRC_VAL_IMG))
random.shuffle(all_val_images)

selected_val_images = all_val_images[:VAL_LIMIT]

copied = 0

for img_name in selected_val_images:
    img_path = os.path.join(SRC_VAL_IMG, img_name)
    lbl_name = img_name.replace(".jpg", ".txt")
    lbl_path = os.path.join(SRC_VAL_LBL, lbl_name)

    new_labels = []

    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                orig_cls = int(parts[0])
                if orig_cls in CLASS_MAP:
                    new_cls = CLASS_MAP[orig_cls]
                    new_labels.append(
                        f"{new_cls} {' '.join(parts[1:])}"
                    )

    # copy image
    shutil.copy(img_path, os.path.join(DST_VAL_IMG, img_name))

    # write new label file
    with open(os.path.join(DST_VAL_LBL, lbl_name), "w") as f:
        for l in new_labels:
            f.write(l + "\n")

    copied += 1

print(f"Validation images copied: {copied}")

In [ ]:
len(os.listdir(DST_VAL_IMG)), len(os.listdir(DST_VAL_LBL))


In [ ]:
import os

data_yaml_path = "./output/road_yolo/data.yaml"

# Use absolute path so YOLO can resolve it regardless of working directory
abs_road_yolo = os.path.abspath("./output/road_yolo").replace("\\", "/")

data_yaml_content = f"""
path: {abs_road_yolo}
train: images/train
val: images/val

names:
  0: crack
  1: pothole
  2: patch
"""

with open(data_yaml_path, "w") as f:
    f.write(data_yaml_content.strip())

print("data.yaml created at:", data_yaml_path)
print("Absolute dataset path:", abs_road_yolo)


In [ ]:
# verify counts
print("Train images:", len(os.listdir(DST_TRAIN_IMG)))
print("Train labels:", len(os.listdir(DST_TRAIN_LBL)))
print("Val images:", len(os.listdir(DST_VAL_IMG)))
print("Val labels:", len(os.listdir(DST_VAL_LBL)))

# check data.yaml
with open(data_yaml_path, "r") as f:
    print("\n--- data.yaml ---")
    print(f.read())


In [ ]:
!pip install -U ultralytics


In [ ]:
from ultralytics import YOLO
import torch

print("Torch CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))


In [ ]:
import os

model = YOLO("yolov8m.pt")

model.train(
    data=os.path.abspath("./output/road_yolo/data.yaml"),
    epochs=250,
    imgsz=1024,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,
    patience=50,
    cos_lr=True,
    workers=2,
    device=0
)


In [ ]:
import os
# YOLO saves training results to ./runs/detect/train/ by default
print(os.listdir("./runs/detect/train/weights"))


In [ ]:
import os
print(os.listdir("./runs/detect"))


In [ ]:
import os
print(os.listdir("."))
